In [3]:
from dotenv import load_dotenv
from pprint import pprint
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage

load_dotenv()

True

In [4]:
client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()
pprint(tools)

[StructuredTool(name='search-flight', description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just city names.\n\nYou should display the returned results in a markdown table format: Group the results by price (those who are the cheapest), duration (those who are the shortest, i.e. have the smallest \'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates 

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt="You are a travel agent. No follow up questions."
)

In [6]:

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from Houston to Sai Gon on Jan 31st, 2027")]}
)

In [7]:
pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from Houston to Sai Gon on Jan 31st, 2027', additional_kwargs={}, response_metadata={}, id='e05446bf-df93-487c-bbe7-1ed5df890670'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 647, 'prompt_tokens': 1233, 'total_tokens': 1880, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DWB5lKA2rppI6Jbd5warDYEG5ly3y', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da351-41d7-7241-85c1-7b73f4efa411-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'Houston', 'flyTo': 'Sai Gon', 'departureDate': '31/01/2027', 'passengers': {'adults': 1, '

In [8]:
print(response["messages"][-1].content)

Direct flights from Houston (IAH) to Ho Chi Minh City (SGN) on Jan 31, 2027 were not found. Here are the best one-stop options on that date (sorted by price and usefulness):

- Cheapest option
  - Route: Houston IAH → Los Angeles LAX → Hong Kong HKG → Ho Chi Minh SGN
  - Duration: approximately 36 hours 20 minutes total
  - Price: USD 731
  - Link: https://on.kiwi.com/rvkuSp

- Shortest-duration option
  - Route: Houston IAH → Tokyo Haneda HND → Ho Chi Minh SGN
  - Duration: approximately 22 hours 30 minutes total
  - Price: USD 779
  - Link: https://on.kiwi.com/fgfQ4n

- Mid-range option
  - Route: Houston IAH → San Francisco SFO → Tokyo Narita NRT → Ho Chi Minh SGN
  - Duration: approximately 28–30 hours total
  - Price: USD 738
  - Link: https://on.kiwi.com/eGproD

Fun fact about the destination: Ho Chi Minh City (aka Saigon) is famous for its vibrant street food scene and iconic coffee culture. A quick tip: trying a traditional Vietnamese coffee with condensed milk while exploring 